In [21]:
import pandas as pd
from pyspark.sql import SparkSession

from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, Binarizer, OneHotEncoder, VectorAssembler, PCA

spark = SparkSession.builder.appName("FinalProject").getOrCreate()

# 1. Read training data

In [22]:
#Instruction: You should read this data into a standard pandas data frame using the pd.read_csv() function.
df_pd = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
df_pd.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [23]:
#Instruction: Convert this to a spark data frame
spark_df = spark.createDataFrame(df_pd)
spark_df.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



In [24]:
spark_df.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

# 2. Pre-processing

Instruction:
We are going to treat the Power_Zone_3 variable as our response variable.
We can use all of the other variables as predictors. (Imagine we know that the Power_Zone_3 reading
is going to go offline in the future and we need to be able to predict that value appropriately.)

In [25]:
#cast hour and rename response variable
sql_transformer = SQLTransformer(statement = """
    SELECT *, CAST(Hour AS DOUBLE) AS Hour_double, Power_Zone_3 AS label
    FROM __THIS__""")
#예측변수인 Power_Zone_3를 label이라는 이름으로 복사

Instruction: The Hour column is likely not stored as a DoubleType. If it is not, use an SQL transformer to cast the
variable as a DoubleType.
Binarize the Hour column based on the column being less than 6.5 or not (night vs day essentially)

In [26]:
#create new binary variable
hour_binarizer = Binarizer(threshold = 6.5, inputCol = "Hour_double", outputCol = "Hour_binary")

Instruction: One-hot encode the Month column

In [28]:
month_encoder = OneHotEncoder(inputCols = ["Month"], outputCols = ["Month_encoded"])

Instruction: Run a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and
Diffuse_Flows columns.

In [31]:
#Instruction: Use a VectorAssembler() call to place these variables in a column together for use with the PCA() estimator.
pca_assembler = VectorAssembler(
    inputCols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"],
    outputCol = "pca_input")

In [32]:
#Instruction: We’ll use two PCs in our transformation.
pca = PCA(k = 2, inputCol = "pca_input", outputCol = "pca_features") #define PCA transformer